# 02 TranFormData

Notebook นี้ใช้สำหรับแปลงข้อมูลจากไฟล์ raw ก่อนเข้าสู่ขั้น clean data

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd



## Load Raw Data

ใช้ไฟล์ raw `vwTimeStampDashbaord.csv` เป็นจุดเริ่มต้น แล้วค่อยแปลงค่าใน notebook นี้

In [2]:
source_path = '../../data/raw/vwTimeStampDashboard.csv'
df = pd.read_csv(source_path, encoding='utf-8-sig')

print(f"source: {source_path}")
print(f"shape: {df.shape}")
df.head()


source: ../../data/raw/vwTimeStampDashboard.csv
shape: (32505, 40)


,PlantName,PickListType,PickDate,TruckSeqNo,CarType,CarNo,PackListNo,CustomerName,QueueTime,PrepareForward,...,ACCESSORIESSapAmount,TruckOverTimeName,TruckOverTimeRemark,TileStart,TileEnd,FittingStart,FittingEnd,AccStart,AccEnd,Unnamed: 39
0,SB1,Walk-in,2025-01-07 11:10:04,8.0,1.000000e+09,89-0471,SB1PL250107022,บ.จำหน่ายวัตถุก่อสร้าง จก. สาขา 3,2025-01-07 11:10:09,N,...,0.0,NaN,NaN,2025-01-07 11:13:27,2025-01-07 11:14:50,2025-01-07 11:11:44,2025-01-07 11:12:30,NaN,NaN,NaN
1,SB1,Walk-in,2025-02-03 15:17:08,26.0,1.000000e+09,บห 3671,SB1PL250203089,PO-LH-250102018 - ศุภาลัย ปาล์ม สปร ศุภาลัย ปา...,2025-02-03 15:17:17,N,...,0.0,NaN,NaN,2025-02-03 15:22:35,2025-02-03 15:24:32,NaN,NaN,NaN,NaN,NaN
2,SB1,Walk-in,2025-01-10 11:11:25,7.0,1.000000e+09,60-6947,SB1PL250110028,SCGR Nakhon Srithammarat Plant SCG Roofing Co....,2025-01-09 16:54:47,N,...,0.0,NaN,NaN,2025-01-10 11:18:11,2025-01-10 11:18:11,2025-01-10 11:16:54,2025-01-10 11:18:11,NaN,NaN,NaN
3,SB1,Walk-in,2025-05-09 16:02:58,7.0,1.000000e+09,71-9972,SB1PL250509064,บริษัท วาลอฟท์ ดีไซน์ จำกัด,2025-05-09 16:03:16,N,...,40.0,NaN,NaN,2025-05-09 17:38:03,2025-05-09 17:43:56,2025-05-09 16:07:58,2025-05-09 16:11:13,2025-05-09 16:11:29,2025-05-09 16:21:55,NaN
4,SB1,SmartQ,2025-04-05 14:15:01,18.0,1.000000e+09,70-4568,SB1PL250405063,คุณแน็ท ท่ามะขาม,2024-12-11 14:39:32,Y,...,23.0,NaN,NaN,2025-04-05 14:44:55,2025-04-05 15:03:58,2025-04-05 14:07:50,2025-04-05 14:13:43,2025-04-05 14:14:39,2025-04-05 14:20:30,NaN


## Quick Check in Raw File

ดูค่าที่น่าสงสัยในไฟล์ raw ก่อนแปลงข้อมูล

In [3]:
for col in ['PickListType', 'PrepareForward', 'CarType', 'TruckStatus', 'PackListStatus']:
    print(f"\n{col}")
    print(df[col].astype(str).value_counts(dropna=False).head(10))


PickListType
PickListType
Walk-in        31244
SmartQ          1260
45926.53397        1
Name: count, dtype: int64

PrepareForward
PrepareForward
N              18557
Y              13940
45681.4207         1
nan                1
46135.63981        1
46106.71063        1
46106.71279        1
46098.60498        1
46106.72737        1
46106.71925        1
Name: count, dtype: int64

CarType
CarType
1000000003.0    16585
1000000005.0    10739
1000000008.0     3580
1000000004.0      947
1000000006.0      202
1000000009.0      194
1000000001.0       98
1000000000.0       79
1000000002.0       50
6.0                15
Name: count, dtype: int64

TruckStatus
TruckStatus
Loading        32497
45681.42567        1
nan                1
46135.69685        1
46111.54631        1
46111.50123        1
46098.61959        1
46111.69963        1
46111.66447        1
Name: count, dtype: int64

PackListStatus
PackListStatus
OPERATORCOMPLETED    32497
Loading                  7
nan                      1
Na

In [4]:
df.loc[
    df['PackListNo'].isna(),
    ['PickDate', 'CarNo', 'PackListNo', 'CustomerName', 'QueueTime', 'PrepareForward', 'TruckStatus', 'PackListStatus']
].head(10)

,PickDate,CarNo,PackListNo,CustomerName,QueueTime,PrepareForward,TruckStatus,PackListStatus
23120,2026-03-30 12:37:52,72-2248 ตู้3/ฟิลิปปินส์,NaN,SB1PL260325070,บริษัท เอสซีจี เทรดดิ้ง จำกัด,46106.71063,46111.54631,Loading
23632,2026-03-30 11:17:33,700-0959 ตู้4/ฟิลิปปินส์,NaN,SB1PL260325071,บริษัท เอสซีจี เทรดดิ้ง จำกัด,46106.71279,46111.50123,Loading
24339,2026-03-17 14:31:02,70-1312 ชลบุรี,NaN,SB1PL260317049,"SCGR Cholburi Plant SCG Roofing Co., Ltd.",46098.60498,46098.61959,Loading
26781,2026-03-30 16:22:06,700-8556 ตู้8/ฟิลิปปินส์,NaN,SB1PL260325076,บริษัท เอสซีจี เทรดดิ้ง จำกัด,46106.72737,46111.69963,Loading
32114,2026-03-30 15:18:18,701-0738 ตู้6/ฟิลิปปินส์,NaN,SB1PL260325073,บริษัท เอสซีจี เทรดดิ้ง จำกัด,46106.71925,46111.66447,Loading


## Mapping Rules

แปลงค่าของ `CarType` และ `PickListType` กลับลงคอลัมน์เดิม

In [5]:
CAR_TYPE_MAP = {
    4: '4 ล้อ', 1000000008: '4 ล้อ',
    6: '6 ล้อ', 1000000003: '6 ล้อ',
    10: '10 ล้อ', 1000000000: '10 ล้อ', 1000000004: '10 ล้อ',
    18: 'เทรเลอร์', 22: 'เทรเลอร์', 1000000001: 'เทรเลอร์',
    1000000002: 'เทรเลอร์', 1000000005: 'เทรเลอร์', 1000000006: 'เทรเลอร์',
    1000000007: 'เทรเลอร์', 1000000009: 'เทรเลอร์',
    1000000010: 'อื่นๆ',
}


def map_pick_list_type(row):
    if row.get('PickListType') == 'SmartQ':
        return 'SmartQ'
    if row.get('PickListType') == 'Walk-in':
        return 'ล่วงหน้า' if row.get('PrepareForward') == 'Y' else 'Walk in'
    return 'Unknown'


def map_car_type(value):
    try:
        return CAR_TYPE_MAP.get(int(value), 'Unknown') if pd.notna(value) else 'Unknown'
    except (TypeError, ValueError):
        return 'Unknown'

## Apply Category Transform

แทนค่าใน `CarType` และ `PickListType` แล้วคงข้อมูลอื่นไว้เหมือนเดิม

In [6]:
df_transform = df.copy()

df_transform['CarType'] = df_transform['CarType'].apply(map_car_type)
df_transform['PickListType'] = df_transform.apply(map_pick_list_type, axis=1)

df_transform[['CarType', 'PickListType', 'PrepareForward']].head(10)

,CarType,PickListType,PrepareForward
0,6 ล้อ,Walk in,N
1,4 ล้อ,Walk in,N
2,เทรเลอร์,Walk in,N
3,6 ล้อ,Walk in,N
4,6 ล้อ,SmartQ,Y
5,6 ล้อ,Walk in,N
6,6 ล้อ,Walk in,N
7,6 ล้อ,ล่วงหน้า,Y
8,เทรเลอร์,ล่วงหน้า,Y
9,6 ล้อ,Walk in,N


## Normalize Missing-like Values

แปลงค่าอย่าง `NULL`, ช่องว่าง, และ `nan` ให้เป็น `NaN` มาตรฐาน

In [7]:
NULL_LIKE_VALUES = ['NULL', '', 'nan', 'NaN', 'None']
df_transform = df_transform.replace(NULL_LIKE_VALUES, np.nan)

df_transform.isna().sum().sort_values(ascending=False).head(15)

Unnamed: 39            32504
TruckOverTimeRemark    32382
TruckOverTimeName      32352
AccEnd                 16956
AccStart               16951
FittingStart           14237
FittingEnd             14236
TileStart               5516
TileEnd                 5511
FirstPostPallet          328
LastPostPallet           328
QueueTime                120
PostingTime               10
PackListNo                 5
CustomerName               1
dtype: int64

## Convert Datetime and Numeric Columns

แปลงคอลัมน์เวลาเป็น `datetime` และคอลัมน์ตัวเลขเป็น `numeric`

In [8]:
datetime_columns = [
    'PickDate', 'QueueTime', 'CreateDate', 'PickingTime', 'OperatorCarConfirm',
    'CarConfirm', 'PostingTime', 'FirstPostPallet', 'LastPostPallet',
    'TileStart', 'TileEnd', 'FittingStart', 'FittingEnd', 'AccStart', 'AccEnd', 'TruckReceiveDate'
]

numeric_columns = [
    'TruckSeqNo', 'TruckReceiveHour', 'TruckReceiveMinute',
    'CPACTileSapAmount', 'PRESTIGETileSapAmount', 'NEUSTILETileSapAmount',
    'CPACFittingSapAmount', 'PRESTIGEFittingSapAmount', 'NEUSTILEFittingSapAmount',
    'DURAFittingSapAmount', 'ACCESSORIESSapAmount'
]

int64_columns = ['CPACTileSapAmount']

for col in datetime_columns:
    df_transform[col] = pd.to_datetime(
        df_transform[col],
        format='%Y-%m-%d %H:%M:%S',
        errors='coerce'
    )

for col in numeric_columns:
    df_transform[col] = pd.to_numeric(df_transform[col], errors='coerce')

for col in int64_columns:
    df_transform[col] = df_transform[col].astype('Int64')

df_transform.dtypes

PlantName                           object
PickListType                        object
PickDate                    datetime64[ns]
TruckSeqNo                         float64
CarType                             object
CarNo                               object
PackListNo                          object
CustomerName                        object
QueueTime                   datetime64[ns]
PrepareForward                      object
TruckReceiveDate            datetime64[ns]
TruckReceiveHour                   float64
TruckReceiveMinute                 float64
CreateDate                  datetime64[ns]
PickingTime                 datetime64[ns]
OperatorCarConfirm          datetime64[ns]
CarConfirm                  datetime64[ns]
PostingTime                 datetime64[ns]
FirstPostPallet             datetime64[ns]
LastPostPallet              datetime64[ns]
TruckStatus                         object
PackListStatus                      object
PostLocationName                    object
CPACTileSap

## Check Converted Data

สรุป dtype หลังแปลง และเช็กจำนวนข้อมูลที่ parse ได้

In [9]:
dtype_summary = pd.DataFrame(
    {
        'column': df_transform.columns,
        'dtype': df_transform.dtypes.astype(str).values,
    }
)

dtype_summary.sort_values(['dtype', 'column']).reset_index(drop=True)

,column,dtype
0,CPACTileSapAmount,Int64
1,AccEnd,datetime64[ns]
2,AccStart,datetime64[ns]
3,CarConfirm,datetime64[ns]
4,CreateDate,datetime64[ns]
5,FirstPostPallet,datetime64[ns]
6,FittingEnd,datetime64[ns]
7,FittingStart,datetime64[ns]
8,LastPostPallet,datetime64[ns]
9,OperatorCarConfirm,datetime64[ns]


In [10]:
datetime_check = pd.DataFrame(
    {
        'column': datetime_columns,
        'non_null_after_parse': [df_transform[col].notna().sum() for col in datetime_columns],
        'missing_after_parse': [df_transform[col].isna().sum() for col in datetime_columns],
    }
)

datetime_check

,column,non_null_after_parse,missing_after_parse
0,PickDate,32505,0
1,QueueTime,32378,127
2,CreateDate,32504,1
3,PickingTime,32504,1
4,OperatorCarConfirm,32504,1
5,CarConfirm,32504,1
6,PostingTime,32495,10
7,FirstPostPallet,32177,328
8,LastPostPallet,32177,328
9,TileStart,26989,5516


In [11]:
numeric_check = pd.DataFrame(
    {
        'column': numeric_columns,
        'non_null_after_parse': [df_transform[col].notna().sum() for col in numeric_columns],
        'missing_after_parse': [df_transform[col].isna().sum() for col in numeric_columns],
    }
)

numeric_check

,column,non_null_after_parse,missing_after_parse
0,TruckSeqNo,32505,0
1,TruckReceiveHour,32504,1
2,TruckReceiveMinute,32504,1
3,CPACTileSapAmount,32497,8
4,PRESTIGETileSapAmount,32504,1
5,NEUSTILETileSapAmount,32504,1
6,CPACFittingSapAmount,32504,1
7,PRESTIGEFittingSapAmount,32504,1
8,NEUSTILEFittingSapAmount,32504,1
9,DURAFittingSapAmount,32504,1


In [12]:
df_transform.head()

,PlantName,PickListType,PickDate,TruckSeqNo,CarType,CarNo,PackListNo,CustomerName,QueueTime,PrepareForward,...,ACCESSORIESSapAmount,TruckOverTimeName,TruckOverTimeRemark,TileStart,TileEnd,FittingStart,FittingEnd,AccStart,AccEnd,Unnamed: 39
0,SB1,Walk in,2025-01-07 11:10:04,8.0,6 ล้อ,89-0471,SB1PL250107022,บ.จำหน่ายวัตถุก่อสร้าง จก. สาขา 3,2025-01-07 11:10:09,N,...,0.0,NaN,NaN,2025-01-07 11:13:27,2025-01-07 11:14:50,2025-01-07 11:11:44,2025-01-07 11:12:30,NaT,NaT,NaN
1,SB1,Walk in,2025-02-03 15:17:08,26.0,4 ล้อ,บห 3671,SB1PL250203089,PO-LH-250102018 - ศุภาลัย ปาล์ม สปร ศุภาลัย ปา...,2025-02-03 15:17:17,N,...,0.0,NaN,NaN,2025-02-03 15:22:35,2025-02-03 15:24:32,NaT,NaT,NaT,NaT,NaN
2,SB1,Walk in,2025-01-10 11:11:25,7.0,เทรเลอร์,60-6947,SB1PL250110028,SCGR Nakhon Srithammarat Plant SCG Roofing Co....,2025-01-09 16:54:47,N,...,0.0,NaN,NaN,2025-01-10 11:18:11,2025-01-10 11:18:11,2025-01-10 11:16:54,2025-01-10 11:18:11,NaT,NaT,NaN
3,SB1,Walk in,2025-05-09 16:02:58,7.0,6 ล้อ,71-9972,SB1PL250509064,บริษัท วาลอฟท์ ดีไซน์ จำกัด,2025-05-09 16:03:16,N,...,40.0,NaN,NaN,2025-05-09 17:38:03,2025-05-09 17:43:56,2025-05-09 16:07:58,2025-05-09 16:11:13,2025-05-09 16:11:29,2025-05-09 16:21:55,NaN
4,SB1,SmartQ,2025-04-05 14:15:01,18.0,6 ล้อ,70-4568,SB1PL250405063,คุณแน็ท ท่ามะขาม,2024-12-11 14:39:32,Y,...,23.0,NaN,NaN,2025-04-05 14:44:55,2025-04-05 15:03:58,2025-04-05 14:07:50,2025-04-05 14:13:43,2025-04-05 14:14:39,2025-04-05 14:20:30,NaN


## Save Transformed Data

บันทึกไฟล์ที่แปลงค่าแล้วไว้ใน `data/interim` เพื่อใช้ต่อในขั้น clean

## Sort by OperatorCarConfirm

จัดเรียงข้อมูลตามวันเวลาที่รถเข้าโรงงาน (`OperatorCarConfirm`) จากเก่าไปใหม่

In [13]:
df_transform = df_transform.sort_values('OperatorCarConfirm', ascending=True, na_position='last').reset_index(drop=True)

print(f"shape: {df_transform.shape}")
df_transform[['OperatorCarConfirm', 'CarNo', 'PlantName']].head(10)

shape: (32505, 40)


,OperatorCarConfirm,CarNo,PlantName
0,2025-01-02 07:18:57,71-4711,SB1
1,2025-01-02 07:21:45,71-3545,SB1
2,2025-01-02 10:52:36,89-1359,SB1
3,2025-01-02 12:05:57,71-6663,SB1
4,2025-01-02 13:12:27,70-6399,SB1
5,2025-01-02 13:13:09,73-0454,SB1
6,2025-01-02 13:16:44,71-0596,SB1
7,2025-01-02 13:39:11,70-9967,SB1
8,2025-01-02 13:40:04,72-5853,SB1
9,2025-01-02 14:22:33,69-7181,SB1


## Re-number TruckSeqNo per Day

กำหนด `TruckSeqNo` ใหม่ภายในแต่ละวัน (แยกตาม `PlantName` + วันที่ของ `OperatorCarConfirm`)  
โดยเรียงลำดับตามเวลา `OperatorCarConfirm` จากน้อยไปมาก เริ่มต้นที่ 1

In [14]:
df_transform['OperatorCarConfirm_Date'] = df_transform['OperatorCarConfirm'].dt.date

df_transform['TruckSeqNo'] = (
    df_transform
    .groupby(['PlantName', 'OperatorCarConfirm_Date'], sort=False)['OperatorCarConfirm']
    .rank(method='first', ascending=True, na_option='bottom')
    .astype('Int64')
)

df_transform = df_transform.drop(columns=['OperatorCarConfirm_Date'])

# ตรวจสอบ
sample_date = df_transform['OperatorCarConfirm'].dt.date.dropna().iloc[0]
check = df_transform[df_transform['OperatorCarConfirm'].dt.date == sample_date][
    ['PlantName', 'TruckSeqNo', 'OperatorCarConfirm', 'CarNo']
].sort_values('TruckSeqNo')
print(f"ตัวอย่างวันที่: {sample_date}")
check

ตัวอย่างวันที่: 2025-01-02


,PlantName,TruckSeqNo,OperatorCarConfirm,CarNo
0,SB1,1,2025-01-02 07:18:57,71-4711
1,SB1,2,2025-01-02 07:21:45,71-3545
2,SB1,3,2025-01-02 10:52:36,89-1359
3,SB1,4,2025-01-02 12:05:57,71-6663
4,SB1,5,2025-01-02 13:12:27,70-6399
5,SB1,6,2025-01-02 13:13:09,73-0454
6,SB1,7,2025-01-02 13:16:44,71-0596
7,SB1,8,2025-01-02 13:39:11,70-9967
8,SB1,9,2025-01-02 13:40:04,72-5853
9,SB1,10,2025-01-02 14:22:33,69-7181


## Encode Categorical Columns

แปลง categorical columns ให้เป็นตัวเลขด้วย mapping ที่กำหนดไว้ล่วงหน้า เพื่อให้ทุก notebook ถัดไปได้ข้อมูล numeric ครบ

| Column | Mapping |
|---|---|
| `CarType` | 4 ล้อ=0, 6 ล้อ=1, 10 ล้อ=2, เทรเลอร์=3 |
| `PickListType` | Walk in=0, SmartQ=1, ล่วงหน้า=2 |
| `PrepareForward` | N=0, Y=1 |
| `PostLocationName` | เรียงตามชื่อ → 0, 1, 2, ... |

In [15]:
CAR_TYPE_ENCODE    = {'4 ล้อ': 0, '6 ล้อ': 1, '10 ล้อ': 2, 'เทรเลอร์': 3}
PICK_LIST_ENCODE   = {'Walk in': 0, 'SmartQ': 1, 'ล่วงหน้า': 2}
PREPARE_FWD_ENCODE = {'N': 0, 'Y': 1}

df_transform['CarType']        = df_transform['CarType'].map(CAR_TYPE_ENCODE).astype('Int64')
df_transform['PickListType']   = df_transform['PickListType'].map(PICK_LIST_ENCODE).astype('Int64')
df_transform['PrepareForward'] = df_transform['PrepareForward'].map(PREPARE_FWD_ENCODE).astype('Int64')

# PostLocationName: เรียง unique values แล้ว map → 0, 1, 2, ...
# ใช้ is_numeric_dtype แทน dtype==object เพราะ pandas ใหม่ return StringDtype ไม่ใช่ object
if not pd.api.types.is_numeric_dtype(df_transform['PostLocationName']):
    loc_cats = sorted(df_transform['PostLocationName'].dropna().unique().tolist())
    loc_encode = {v: i for i, v in enumerate(loc_cats)}
    df_transform['PostLocationName'] = df_transform['PostLocationName'].map(loc_encode).astype('Int64')
    print('PostLocationName mapping:')
    for k, v in loc_encode.items():
        print(f'  {v} → {k}')
    print()

# ตรวจสอบผล
encode_cols = ['CarType', 'PickListType', 'PrepareForward', 'PostLocationName']
print('หลัง encode:')
print(df_transform[encode_cols].dtypes)
print()
print('ค่า null หลัง encode (Unknown = NaN):')
print(df_transform[encode_cols].isna().sum())
print()
df_transform[encode_cols].head(10)

PostLocationName mapping:
  0 → OPERATORCOMPLETED
  1 → SB1-1ลานโอน
  2 → SB1ลานโอน
  3 → ลานจ่าย 1 ช่องจ่าย 1 (CPAC)
  4 → ลานจ่าย 1 ช่องจ่าย 2 (CPAC)
  5 → ลานจ่าย 1 ช่องจ่าย 3 (CPAC)
  6 → ลานจ่าย 2 ช่องจ่าย 1 (Prestige)
  7 → ลานจ่าย 2 ช่องจ่าย 2 (Prestige)
  8 → ลานจ่าย 2 ช่องจ่าย 3 (Prestige)
  9 → ลานจ่าย3 ช่องจ่าย 1 (ครอบ)
  10 → ลานจ่าย3 ช่องจ่าย 2 (ครอบ)
  11 → อุปกรณ์

หลัง encode:
CarType             Int64
PickListType        Int64
PrepareForward      Int64
PostLocationName    Int64
dtype: object

ค่า null หลัง encode (Unknown = NaN):
CarType             3
PickListType        1
PrepareForward      8
PostLocationName    1
dtype: int64



,CarType,PickListType,PrepareForward,PostLocationName
0,3,0,0,3
1,3,0,0,3
2,1,0,0,3
3,3,0,0,9
4,1,0,0,7
5,1,0,0,3
6,1,0,0,6
7,2,0,0,6
8,3,2,1,9
9,1,0,0,8


In [16]:
output_path = Path('../../data/interim/vw_timestamp_dashboard_transformed.csv').resolve()
df_transform.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f'Saved → {output_path}')
print(f'Shape  : {df_transform.shape}')

Saved → C:\SCG-Roofing\WMS-ML\WMS-ML\data\interim\vw_timestamp_dashboard_transformed.csv
Shape  : (32505, 40)
